In [ ]:
import pandas as pd

# ===============================
# Q4-1. 재구매 기간 구간 생성 (Pandas)
# ===============================
# SQL에서 생성한 주문 데이터를 기준으로
# 고객별 첫 구매와 두 번째 구매 사이의 기간을 계산하고
# 재구매 기간 구간을 생성하여 Power BI 분석에 활용

# 1. 데이터 로드
file_path = "../data/q4_repurchase_orders_base.csv"
df = pd.read_csv(file_path)

# 2. 구매 순서 생성
df["purchase_date"] = pd.to_datetime(df["purchase_date"])
df = df.sort_values(["customer_unique_id", "purchase_date"])
df["purchase_order"] = df.groupby("customer_unique_id").cumcount() + 1

# 3. 첫 구매 / 두 번째 구매 정보 추출
first_df = df[df["purchase_order"] == 1][["customer_unique_id", "purchase_date"]].rename(
    columns={"purchase_date": "first_purchase_date"}
)
second_df = df[df["purchase_order"] == 2][["customer_unique_id", "purchase_date"]].rename(
    columns={"purchase_date": "second_purchase_date"}
)

repurchase_df = pd.merge(first_df, second_df, on="customer_unique_id")

# 4. 재구매 기간 계산
repurchase_df["repurchase_days"] = (
    repurchase_df["second_purchase_date"] - repurchase_df["first_purchase_date"]
).dt.days
repurchase_df = repurchase_df[repurchase_df["repurchase_days"] > 0]

# 5. 재구매 기간 구간 생성
bins = [0, 10, 30, 90, float("inf")]
labels = ["1~10일", "11~30일", "31~90일", "91일 이상"]

repurchase_df["repurchase_bucket"] = pd.cut(
    repurchase_df["repurchase_days"],
    bins=bins,
    labels=labels
)

repurchase_df.head()

# 6. 결과 저장
output_path = "../data/q4_repurchase_orders_pandas.csv"
repurchase_df.to_csv(output_path, index=False, encoding="utf-8-sig")

repurchase_bucket_new
1~30일     587
31~60일    296
61~90일    196
91일 이상    893
Name: count, dtype: int64
